In [1]:
import numpy as np
import urllib.request

urllib.request.urlretrieve(
    "https://www.fredjo.com/files/ihdp_npci_1-100.train.npz", "ihdp_train.npz")
urllib.request.urlretrieve(
    "https://www.fredjo.com/files/ihdp_npci_1-100.test.npz", "ihdp_test.npz")

train = np.load("ihdp_train.npz")
test = np.load("ihdp_test.npz")

print("Keys:", train.files)
print("x shape (covariates):", train['x'].shape)   # expect (747, 25, 100) — 100 realizations
print("t shape (treatment):", train['t'].shape)
print("yf shape (factual outcome):", train['yf'].shape)
print("mu0, mu1 shape (true potential outcomes):", train['mu0'].shape, train['mu1'].shape)

# Check realization 0 specifically, since that's what we'll fit first
x0, t0, yf0 = train['x'][:, :, 0], train['t'][:, 0], train['yf'][:, 0]
mu0_0, mu1_0 = train['mu0'][:, 0], train['mu1'][:, 0]

true_ite = mu1_0 - mu0_0
true_ate = true_ite.mean()
print(f"\nRealization 0 — true ATE: {true_ate:.4f}")
print(f"Treated: {t0.sum():.0f}, Control: {(1-t0).sum():.0f}")
print(f"True ITE range: [{true_ite.min():.2f}, {true_ite.max():.2f}]")

Keys: ['ate', 'mu1', 'mu0', 'yadd', 'yf', 'ycf', 't', 'x', 'ymul']
x shape (covariates): (672, 25, 100)
t shape (treatment): (672, 100)
yf shape (factual outcome): (672, 100)
mu0, mu1 shape (true potential outcomes): (672, 100) (672, 100)

Realization 0 — true ATE: 4.0116
Treated: 123, Control: 549
True ITE range: [-1.87, 4.67]


In [2]:
!pip install econml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 750.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [3]:
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

x, t, yf = train['x'][:, :, 0], train['t'][:, 0], train['yf'][:, 0]
mu0, mu1 = train['mu0'][:, 0], train['mu1'][:, 0]
true_ite = mu1 - mu0
true_ate = true_ite.mean()

# Naive difference-in-means (uncorrected upper bound, same role as your Pearson baseline)
naive_ate = yf[t == 1].mean() - yf[t == 0].mean()

# Linear DML (your Section 3.7 baseline)
ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                  model_t=GradientBoostingRegressor(random_state=42),
                  cv=5, random_state=42)
ldml.fit(yf, t, X=x)
ldml_ate = ldml.ate(x)

# CausalForestDML (your proposed estimator)
cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=GradientBoostingRegressor(random_state=42),
                      n_estimators=1000, min_samples_leaf=10, honest=True,
                      cv=5, random_state=42)
cf.fit(yf, t, X=x)
cf_ate = cf.ate(x)
cf_ite = cf.effect(x)

pehe = np.sqrt(np.mean((cf_ite - true_ite) ** 2))
ate_err_naive = abs(naive_ate - true_ate)
ate_err_ldml = abs(ldml_ate - true_ate)
ate_err_cf = abs(cf_ate - true_ate)

print(f"True ATE:              {true_ate:.4f}")
print(f"Naive diff-in-means:   {naive_ate:.4f}  (abs error: {ate_err_naive:.4f})")
print(f"Linear DML:            {ldml_ate:.4f}  (abs error: {ate_err_ldml:.4f})")
print(f"CausalForestDML:       {cf_ate:.4f}  (abs error: {ate_err_cf:.4f})")
print(f"\nPEHE (sqrt mean squared ITE error): {pehe:.4f}")

True ATE:              4.0116
Naive diff-in-means:   4.0047  (abs error: 0.0069)
Linear DML:            3.8795  (abs error: 0.1320)
CausalForestDML:       3.8681  (abs error: 0.1435)

PEHE (sqrt mean squared ITE error): 0.6174


In [4]:
import numpy as np
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import GradientBoostingRegressor

n_reps = 10  # standard first pass; scale to 100 once this runs cleanly

results = {'naive_ate_err': [], 'ldml_ate_err': [], 'cf_ate_err': [],
           'ldml_pehe': [], 'cf_pehe': []}

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    naive_ate = yf[t == 1].mean() - yf[t == 0].mean()

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=GradientBoostingRegressor(random_state=42),
                      cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)

    cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                          model_t=GradientBoostingRegressor(random_state=42),
                          n_estimators=1000, min_samples_leaf=10, honest=True,
                          cv=5, random_state=42)
    cf.fit(yf, t, X=x)
    cf_ite = cf.effect(x)

    results['naive_ate_err'].append(abs(naive_ate - true_ate))
    results['ldml_ate_err'].append(abs(ldml_ite.mean() - true_ate))
    results['cf_ate_err'].append(abs(cf_ite.mean() - true_ate))
    results['ldml_pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))
    results['cf_pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    print(f"Realization {i}: naive_err={results['naive_ate_err'][-1]:.3f}, "
          f"ldml_err={results['ldml_ate_err'][-1]:.3f}, cf_err={results['cf_ate_err'][-1]:.3f}, "
          f"ldml_pehe={results['ldml_pehe'][-1]:.3f}, cf_pehe={results['cf_pehe'][-1]:.3f}")

print("\n--- Summary across", n_reps, "realizations ---")
for k, v in results.items():
    print(f"{k}: {np.mean(v):.4f} ± {np.std(v):.4f}")

Realization 0: naive_err=0.007, ldml_err=0.132, cf_err=0.143, ldml_pehe=0.659, cf_pehe=0.617
Realization 1: naive_err=0.026, ldml_err=0.162, cf_err=0.356, ldml_pehe=0.982, cf_pehe=0.690
Realization 2: naive_err=0.048, ldml_err=0.261, cf_err=0.193, ldml_pehe=0.831, cf_pehe=0.560
Realization 3: naive_err=0.011, ldml_err=0.511, cf_err=0.505, ldml_pehe=0.896, cf_pehe=1.513
Realization 4: naive_err=0.053, ldml_err=0.347, cf_err=0.267, ldml_pehe=1.238, cf_pehe=1.935
Realization 5: naive_err=0.109, ldml_err=0.282, cf_err=0.270, ldml_pehe=0.964, cf_pehe=0.656
Realization 6: naive_err=0.253, ldml_err=0.468, cf_err=0.362, ldml_pehe=0.935, cf_pehe=0.446
Realization 7: naive_err=0.173, ldml_err=0.463, cf_err=0.181, ldml_pehe=1.339, cf_pehe=0.948
Realization 8: naive_err=1.583, ldml_err=1.363, cf_err=3.668, ldml_pehe=13.070, cf_pehe=17.016
Realization 9: naive_err=0.354, ldml_err=0.542, cf_err=0.700, ldml_pehe=3.564, cf_pehe=6.197

--- Summary across 10 realizations ---
naive_ate_err: 0.2618 ± 0.45

In [5]:
import numpy as np

results_arr = {k: np.array(v) for k, v in results.items()}

print("--- With all 10 realizations ---")
for k, v in results_arr.items():
    print(f"{k}: mean={v.mean():.4f} ± {v.std():.4f}, median={np.median(v):.4f}")

print("\n--- Excluding realization 8 (outlier) ---")
mask = np.ones(10, dtype=bool)
mask[8] = False
for k, v in results_arr.items():
    print(f"{k}: mean={v[mask].mean():.4f} ± {v[mask].std():.4f}, median={np.median(v[mask]):.4f}")

# Also worth knowing: how many realizations does CF actually beat LDML on?
cf_wins_ate = (results_arr['cf_ate_err'] < results_arr['ldml_ate_err']).sum()
cf_wins_pehe = (results_arr['cf_pehe'] < results_arr['ldml_pehe']).sum()
print(f"\nCF beats LDML on ATE error in {cf_wins_ate}/10 realizations")
print(f"CF beats LDML on PEHE in {cf_wins_pehe}/10 realizations")

--- With all 10 realizations ---
naive_ate_err: mean=0.2618 ± 0.4536, median=0.0809
ldml_ate_err: mean=0.4531 ± 0.3323, median=0.4048
cf_ate_err: mean=0.6647 ± 1.0137, median=0.3133
ldml_pehe: mean=2.4478 ± 3.6280, median=0.9728
cf_pehe: mean=3.0578 ± 4.9312, median=0.8191

--- Excluding realization 8 (outlier) ---
naive_ate_err: mean=0.1150 ± 0.1148, median=0.0526
ldml_ate_err: mean=0.3520 ± 0.1433, median=0.3468
cf_ate_err: mean=0.3310 ± 0.1674, median=0.2703
ldml_pehe: mean=1.2676 ± 0.8343, median=0.9638
cf_pehe: mean=1.5070 ± 1.7222, median=0.6899

CF beats LDML on ATE error in 6/10 realizations
CF beats LDML on PEHE in 6/10 realizations


In [6]:
n_reps = 100
results = {'naive_ate_err': [], 'ldml_ate_err': [], 'cf_ate_err': [],
           'ldml_pehe': [], 'cf_pehe': []}

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    naive_ate = yf[t == 1].mean() - yf[t == 0].mean()

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=GradientBoostingRegressor(random_state=42),
                      cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)

    cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                          model_t=GradientBoostingRegressor(random_state=42),
                          n_estimators=1000, min_samples_leaf=10, honest=True,
                          cv=5, random_state=42)
    cf.fit(yf, t, X=x)
    cf_ite = cf.effect(x)

    results['naive_ate_err'].append(abs(naive_ate - true_ate))
    results['ldml_ate_err'].append(abs(ldml_ite.mean() - true_ate))
    results['cf_ate_err'].append(abs(cf_ite.mean() - true_ate))
    results['ldml_pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))
    results['cf_pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 10 == 0:
        print(f"...completed realization {i}")

results_arr = {k: np.array(v) for k, v in results.items()}
print("\n--- Summary across 100 realizations ---")
for k, v in results_arr.items():
    print(f"{k}: mean={v.mean():.4f} ± {v.std():.4f}, median={np.median(v):.4f}")

# Identify heavy-tailed realizations for transparency (same pattern as #8, #9)
outlier_idx = np.where(results_arr['cf_pehe'] > 3 * np.median(results_arr['cf_pehe']))[0]
print(f"\nRealizations with CF PEHE > 3x median (heavy-tailed, likely exponential-surface draws): {outlier_idx.tolist()}")

cf_wins_ate = (results_arr['cf_ate_err'] < results_arr['ldml_ate_err']).sum()
cf_wins_pehe = (results_arr['cf_pehe'] < results_arr['ldml_pehe']).sum()
print(f"CF beats LDML on ATE error in {cf_wins_ate}/100 realizations")
print(f"CF beats LDML on PEHE in {cf_wins_pehe}/100 realizations")

...completed realization 0
...completed realization 10
...completed realization 20
...completed realization 30
...completed realization 40
...completed realization 50
...completed realization 60
...completed realization 70
...completed realization 80
...completed realization 90

--- Summary across 100 realizations ---
naive_ate_err: mean=0.2856 ± 0.3841, median=0.1505
ldml_ate_err: mean=0.4651 ± 0.6850, median=0.3046
cf_ate_err: mean=0.6985 ± 1.1051, median=0.3781
ldml_pehe: mean=2.6855 ± 3.7819, median=1.2615
cf_pehe: mean=3.9215 ± 5.7776, median=1.8165

Realizations with CF PEHE > 3x median (heavy-tailed, likely exponential-surface draws): [8, 9, 12, 20, 25, 27, 33, 36, 38, 52, 59, 67, 70, 80, 81, 83, 84, 85, 92, 97]
CF beats LDML on ATE error in 39/100 realizations
CF beats LDML on PEHE in 26/100 realizations


In [7]:
from sklearn.linear_model import LogisticRegressionCV

n_reps = 100
configs = {
    'leaf10': dict(min_samples_leaf=10, n_estimators=1000),
    'leaf20': dict(min_samples_leaf=20, n_estimators=1000),
    'leaf30': dict(min_samples_leaf=30, n_estimators=1000),
}

all_results = {name: {'ate_err': [], 'pehe': []} for name in configs}
ldml_results = {'ate_err': [], 'pehe': []}
naive_results = []

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    naive_results.append(abs((yf[t==1].mean() - yf[t==0].mean()) - true_ate))

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)
    ldml_results['ate_err'].append(abs(ldml_ite.mean() - true_ate))
    ldml_results['pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    for name, params in configs.items():
        cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                              model_t=LogisticRegressionCV(cv=5, random_state=42),
                              discrete_treatment=True, honest=True, cv=5,
                              random_state=42, **params)
        cf.fit(yf, t, X=x)
        cf_ite = cf.effect(x)
        all_results[name]['ate_err'].append(abs(cf_ite.mean() - true_ate))
        all_results[name]['pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 20 == 0:
        print(f"...completed realization {i}")

print(f"\nNaive: ate_err mean={np.mean(naive_results):.4f}, median={np.median(naive_results):.4f}")
print(f"LDML (discrete_treatment=True): ate_err mean={np.mean(ldml_results['ate_err']):.4f}, "
      f"median={np.median(ldml_results['ate_err']):.4f}, "
      f"pehe mean={np.mean(ldml_results['pehe']):.4f}, median={np.median(ldml_results['pehe']):.4f}")
for name in configs:
    a, p = all_results[name]['ate_err'], all_results[name]['pehe']
    print(f"CF {name}: ate_err mean={np.mean(a):.4f}, median={np.median(a):.4f}, "
          f"pehe mean={np.mean(p):.4f}, median={np.median(p):.4f}")

/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 0


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 20


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...

...completed realization 60


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

...completed realization 80


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 


Naive: ate_err mean=0.2856, median=0.1505
LDML (discrete_treatment=True): ate_err mean=0.5635, median=0.4684, pehe mean=2.7737, median=1.4300
CF leaf10: ate_err mean=0.6354, median=0.3452, pehe mean=3.8760, median=1.8206
CF leaf20: ate_err mean=0.6692, median=0.3594, pehe mean=4.2169, median=1.9471
CF leaf30: ate_err mean=0.6843, median=0.3574, pehe mean=4.4544, median=2.0571


In [8]:
from sklearn.linear_model import LassoCV, LogisticRegressionCV

ldml_results2 = {'ate_err': [], 'pehe': []}
cf_results2 = {'ate_err': [], 'pehe': []}

for i in range(n_reps):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    ldml = LinearDML(model_y=LassoCV(cv=5, random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x)
    ldml_ite = ldml.effect(x)
    ldml_results2['ate_err'].append(abs(ldml_ite.mean() - true_ate))
    ldml_results2['pehe'].append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    cf = CausalForestDML(model_y=LassoCV(cv=5, random_state=42),
                          model_t=LogisticRegressionCV(cv=5, random_state=42),
                          discrete_treatment=True, honest=True, cv=5,
                          min_samples_leaf=10, n_estimators=1000, random_state=42)
    cf.fit(yf, t, X=x)
    cf_ite = cf.effect(x)
    cf_results2['ate_err'].append(abs(cf_ite.mean() - true_ate))
    cf_results2['pehe'].append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

    if i % 20 == 0:
        print(f"...completed realization {i}")

print(f"\nLDML (Lasso/Logistic nuisance): ate_err mean={np.mean(ldml_results2['ate_err']):.4f}, "
      f"median={np.median(ldml_results2['ate_err']):.4f}, "
      f"pehe mean={np.mean(ldml_results2['pehe']):.4f}, median={np.median(ldml_results2['pehe']):.4f}")
print(f"CF (Lasso/Logistic nuisance): ate_err mean={np.mean(cf_results2['ate_err']):.4f}, "
      f"median={np.median(cf_results2['ate_err']):.4f}, "
      f"pehe mean={np.mean(cf_results2['pehe']):.4f}, median={np.median(cf_results2['pehe']):.4f}")

/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 0


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 20


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...

...completed realization 60


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV

...completed realization 80


/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LassoCV


LDML (Lasso/Logistic nuisance): ate_err mean=0.6580, median=0.5259, pehe mean=2.5909, median=1.2863
CF (Lasso/Logistic nuisance): ate_err mean=0.7067, median=0.3889, pehe mean=3.8424, median=1.7430


In [9]:
# Re-run once more, this time storing per-realization arrays for the win-count
# (same config as the "discrete_treatment=True" run — just capturing what wasn't saved)
cf_ate_arr, cf_pehe_arr, ldml_ate_arr, ldml_pehe_arr = [], [], [], []

for i in range(100):
    x, t, yf = train['x'][:, :, i], train['t'][:, i], train['yf'][:, i]
    mu0, mu1 = train['mu0'][:, i], train['mu1'][:, i]
    true_ite = mu1 - mu0
    true_ate = true_ite.mean()

    ldml = LinearDML(model_y=GradientBoostingRegressor(random_state=42),
                      model_t=LogisticRegressionCV(cv=5, random_state=42),
                      discrete_treatment=True, cv=5, random_state=42)
    ldml.fit(yf, t, X=x); ldml_ite = ldml.effect(x)
    ldml_ate_arr.append(abs(ldml_ite.mean() - true_ate))
    ldml_pehe_arr.append(np.sqrt(np.mean((ldml_ite - true_ite) ** 2)))

    cf = CausalForestDML(model_y=GradientBoostingRegressor(random_state=42),
                          model_t=LogisticRegressionCV(cv=5, random_state=42),
                          discrete_treatment=True, honest=True, cv=5,
                          min_samples_leaf=10, n_estimators=1000, random_state=42)
    cf.fit(yf, t, X=x); cf_ite = cf.effect(x)
    cf_ate_arr.append(abs(cf_ite.mean() - true_ate))
    cf_pehe_arr.append(np.sqrt(np.mean((cf_ite - true_ite) ** 2)))

cf_ate_arr, cf_pehe_arr = np.array(cf_ate_arr), np.array(cf_pehe_arr)
ldml_ate_arr, ldml_pehe_arr = np.array(ldml_ate_arr), np.array(ldml_pehe_arr)
print(f"CF beats LDML on ATE error: {(cf_ate_arr < ldml_ate_arr).sum()}/100")
print(f"CF beats LDML on PEHE: {(cf_pehe_arr < ldml_pehe_arr).sum()}/100")

/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: Model LogisticRegressionCV(cv=5, random_state=42) has a non-default cv attribute, which will be ignored
  warnings.warn(f"Model {sub_model} has a non-default cv attribute, which will be ignored")
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/model_selection.py:554: UserWarning: 

CF beats LDML on ATE error: 69/100
CF beats LDML on PEHE: 30/100
